In [1]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime

# today
today = datetime.today().strftime("%y%m%d")

## Pull patents for manual review

In [2]:
# Connect to database
db = duckdb.connect("patents.db")
db.execute("PRAGMA memory_limit='3GB'")
db.execute("PRAGMA temp_directory='C:/temp/duckdb_spill'")
db.execute("PRAGMA threads=2")

In [3]:
# Show tables in database
db.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ patents_classified      │
│ patents_embeddings      │
│ run_260710_1614         │
│ run_260710_1614_reverse │
└─────────────────────────┘

#### Get data from patents_classified

In [3]:
# Create dataframe from classified patents
data = db.sql("SELECT * FROM patents_classified").df()

In [ ]:
# If this is a review on an updated dataset, meaning the columns scope_curated and pillar_curated already exist, perform this filtering step
#data = data[data['scope_curated'].isna()]

In [4]:
data['pred_combined'].value_counts()

pred_combined
in     24046
out     7324
Name: count, dtype: int64

In [11]:
print(f"Of {len(data)} patents, {round(len(data[data["pred_combined"] == "out"])/len(data), 2)*100}% were removed during the ML classification.") 
print(f"\nOf the remaining {len(data[data["pred_combined"] == "in"])} patents, {round(len(data[data["scope_LLM"] == "in"])/len(data[data["pred_combined"] == "in"]), 2)*100}% ({len(data[data["scope_LLM"] == "in"])}) were classified as in scope by the LLM")

Of 10791 patents, 23.0% were removed during the ML classification.

Of the remaining 8294 patents, 37.0% (3063) were classified as in scope by the LLM


#### Create filter mask for curated scope and pillar 

In [5]:
# Scenario 3
has_pillar = (data['pillar_LLM'] != "NA") | (data['pred_pillar'] != "NA")
conf       = data['confidence_LLM']
proba      = data['proba_scope']

conditions = [
    (data['scope_LLM'] == "NA") & (data['pred_combined'] == 'in'),
    (data['scope_LLM'] == 'in') & (conf >= 5),
    (conf <= 4) & (proba > 0.8) & (data['pred_pillar'] != "NA"),
    (data['scope_LLM'] == 'out') & (conf == 2) & (proba < 0.6) & (data['pred_pillar'] == "NA"),
    (data['scope_LLM'] == 'out') & (conf == 1) & (proba < 0.8),
]
choices = ['manual_review', 'in', 'in', 'out', 'out']

#### Assign curated scope/pillar or manual review

In [6]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated'] = np.select(conditions, choices, default='manual_review')
data['scope_curated'] = np.where(data['pred_combined'] == 'out', 'out', data['scope_curated'])
data['pillar_curated'] = np.where(data['scope_curated'] == 'out', np.nan, data['pillar_LLM'].replace('NA', np.nan).fillna(data['pred_pillar'].replace('NA', np.nan)))
data['pillar_curated'] = np.where(data['pred_combined'] == 'out', np.nan, data['pillar_curated'])
data['date_review'] = today

In [7]:
# How many patents have to go through manual review?
data['scope_curated'].value_counts()

scope_curated
out              16283
in               11685
manual_review     3402
Name: count, dtype: int64

In [8]:
data[data['scope_curated'] == "manual_review"]['confidence_LLM'].value_counts()

confidence_LLM
2.0    2440
3.0     369
4.0     103
6.0      55
1.0      16
5.0      12
7.0      10
Name: count, dtype: int64

In [28]:
data[data['scope_curated'] == "manual_review"].loc[:,"proba_scope":"pillar_LLM"]

,proba_scope,pred_combined,pred_pillar,date_ML,scope_LLM,confidence_LLM,pillar_LLM
15,0.781788,in,PB,260710,out,2.0,NA
87,0.634232,in,NA,260710,out,2.0,NA
233,0.431873,in,CC,260710,out,2.0,NA
284,0.457384,in,PB,260710,out,2.0,NA
297,0.474212,in,PB,260710,out,2.0,NA
...,...,...,...,...,...,...,...
10728,0.742741,in,PB,260710,out,2.0,NA
10735,0.791046,in,NA,260710,out,2.0,NA
10751,0.608197,in,NA,260710,out,3.0,NA
10753,0.701477,in,CM,260710,out,2.0,NA


#### Save borderline patents for manual review

In [29]:
# Export patents for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(f"data/patents_for_review_{today}.csv")

In [ ]:
# review = pd.read_csv(f"data/patents_for_review_{today}.csv")

#### Save (automatically) curated scope and pillar back to patents_classified

In [30]:
# filter for the patents with clear scope and pillar assigment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [31]:
# create columns in patents_classified if they don't already exist
db.sql(f"ALTER TABLE patents_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql(f"ALTER TABLE patents_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [32]:
# add to database
db.register('data', data[['id', 'scope_curated', 'pillar_curated']])
db.sql(f"""
    UPDATE patents_classified
    SET scope_curated     = data.scope_curated,
        pillar_curated    = data.pillar_curated
    FROM data
    WHERE patents_classified.id = data.id
""")

In [33]:
db.close()

## Assign manually curated scope and pillar to respective patents

In [21]:
# Connect to database
db = duckdb.connect("patents.db")

In [6]:
# load manually reviewed data
reviewed_data = pd.read_csv(f"data/patents_reviewed_{today}.csv")

#### Add scope and pillar back to original dataset, using publication ID

In [7]:
# add to database
db.register('reviewed_data', reviewed_data[['id', 'scope_curated', 'pillar_curated']])
db.sql(f"""
    UPDATE patents_classified
    SET scope_curated     = reviewed_data.scope_curated,
        pillar_curated    = reviewed_data.pillar_curated
    FROM reviewed_data
    WHERE patents_classified.id = reviewed_data.id
""")

In [8]:
data = db.sql("SELECT * FROM patents_classified").df()

In [9]:
data[~data['scope_curated'].isna()]

,id,title,abstract,application_number,assignee_cities,assignee_countries,assignee_names,cpc,family_count,family_id,...,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,status_LLM,stop_reason_LLM,date_LLM,assignee_names_cleaned,scope_curated,pillar_curated
0,ZA-202311713-B,"PROCESS FOR PRODUCING COOKABLE, FIBROUS MEAT A...",The present disclosure provides a process for ...,ZA2023/11713,"[{""id"": ""6167865"", ""name"": ""Toronto""}]","[{""id"": ""CA"", ""name"": ""Canada""}]","[""NEW SCHOOL FOODS INC""]","[""A23J3/24"", ""A23V2002/00"", ""A23J3/18"", ""A23J3...",13,84104569,...,False,False,False,True,ok,tool_use,260708,[New School Foods],in,CC
1,ZA-202306560-B,MEAT SUBSTITUTE COMPRISING ANIMAL MYOGLOBIN,Described herein is a meat substitute or food ...,ZA2023/06560,"[{""id"": ""2799397"", ""name"": ""Diest""}]","[{""id"": ""BE"", ""name"": ""Belgium""}]","[""PALEO B V""]","[""A23J3/26"", ""C07K14/805"", ""A23L13/424"", ""A23L...",20,79927393,...,False,False,False,False,ok,tool_use,260708,"[[""PALEO B V""]]",out,NA
2,ZA-202408743-B,FISH PROTEIN FOLIAR BIO-ORGANIC FERTILIZER AS ...,The present invention discloses a fish protein...,ZA2024/08743,"[{""id"": ""2037013"", ""name"": ""Harbin""}]","[{""id"": ""CN"", ""name"": ""China""}]","[""HEILONGJIANG GREEN FOOD SCIENCE RES INSTITUT...",NaN,1,96170168,...,False,False,False,False,ok,tool_use,260708,"[[""HEILONGJIANG GREEN FOOD SCIENCE RES INSTITU...",out,NA
3,ZA-202405875-B,METHOD FOR PRODUCING DAIRY SUBSTITUTE PRODUCTS,The present invention relates to a method for ...,ZA2024/05875,NaN,"[{""id"": ""BE"", ""name"": ""Belgium""}]","[""MEURA S A""]","[""A23L11/60"", ""A23L2/74"", ""B01D25/127"", ""A23L2...",10,80735470,...,True,False,False,False,ok,tool_use,260708,"[[""MEURA S A""]]",in,PB
5,ZA-202307189-B,NUTRIENT MEDIA FOR CELL CULTURE CONTAINING PLA...,The present invention relates to nutritional c...,ZA2023/07189,"[{""id"": ""2658209"", ""name"": ""Uzwil""}]","[{""id"": ""CH"", ""name"": ""Switzerland""}]","[""BUEHLER AG""]","[""C12N5/0043"", ""A23J3/14"", ""A23J1/14"", ""A23J3/...",16,74595223,...,False,False,True,False,ok,tool_use,260708,"[[""BUEHLER AG""]]",in,CM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,WO-2025240521-A3,GENETIC FEATURES OF SUSPENSION BLUEFIN TUNA CELLS,Provided herein are altered cell lines compris...,US2025/029203,NaN,"[{'id': US, 'name': United States}]",[Bluenalu Inc],"[C12N2500/99, C12N5/0658, C12N2510/00]",2,96271590,...,False,False,False,False,ok,tool_use,260708,[[BLUENALU INC]],out,NA
264,WO-2025235817-A3,METHOD FOR ENZYMATIC MODIFICATION OF LIQUIFIED...,Described herein are methods for producing oat...,US2025/028497,NaN,"[{'id': US, 'name': United States}]",[World Co Holdings LLC],"[A23L2/38, A23L7/107, A23C11/10]",2,97675778,...,True,False,False,False,ok,tool_use,260708,[[WORLD CO HOLDINGS LLC]],in,PB
265,ZA-202213273-B,MICROBIAL-BASED PROCESS FOR IMPROVED QUALITY P...,The present invention describes a bio-based pr...,ZA2022/13273,"[{'id': 5226534, 'name': Brookings}]","[{'id': US, 'name': United States}]",[PRAIRIE AQUATECH LLC],"[A23K50/80, C12P21/06, A23J3/16, A23J1/148, A2...",27,78845797,...,False,False,False,False,ok,tool_use,260708,[[PRAIRIE AQUATECH LLC]],out,NA
278,WO-2025215354-A1,METHOD OF CULTURING AN ANIMAL CELL,The invention relates to methods for culturing...,GB2025/050744,NaN,NaN,NaN,"['C12N2500/95', 'C12N2500/92', 'C12N2500/99', ...",<NA>,95476380,...,False,False,False,False,ok,tool_use,260708,<NA>,in,CM


In [3]:
db.close()